# At-Risk Learner Prediction Model

Train a GradientBoostingClassifier on synthetic student engagement data to
predict which learners are at risk of falling behind. Includes SHAP
explanations for model interpretability.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from src.prediction.at_risk_model import AtRiskPipeline

## 1. Generate Synthetic Data

In [ ]:
pipeline = AtRiskPipeline()
X, y = pipeline.create_features(n_students=500, seed=42)

df = pd.DataFrame(X, columns=pipeline.FEATURE_NAMES)
df['at_risk'] = y

print(f"Dataset shape: {X.shape}")
print(f"Class balance: {np.bincount(y)}")
print(f"At-risk rate: {y.mean():.2%}")
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(pipeline.FEATURE_NAMES):
    ax = axes[i // 3, i % 3]
    df.groupby('at_risk')[col].hist(alpha=0.6, ax=ax, bins=20, legend=True)
    ax.set_title(col)
    ax.legend(['Not at risk', 'At risk'])
plt.tight_layout()
plt.show()

## 3. Train and Log to MLflow

In [ ]:
pipeline.train_and_log(X, y, experiment_name='at_risk_prediction')
print('Model trained and logged to MLflow.')

## 4. Evaluate on Train Set

In [ ]:
y_pred = pipeline.predict(X)
print(classification_report(y, y_pred, target_names=['Not at risk', 'At risk']))

cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not at risk', 'At risk'])
disp.plot(cmap='Blues')
plt.title('At-Risk Prediction — Confusion Matrix')
plt.show()

## 5. SHAP Explanations

In [ ]:
import shap

# Get the trained GBM model from the pipeline
gbm_model = pipeline.pipeline.named_steps['model']
scaler = pipeline.pipeline.named_steps['scaler']
X_scaled = scaler.transform(X)

explainer = shap.TreeExplainer(gbm_model)
shap_values = explainer.shap_values(X_scaled)

# Summary plot
shap.summary_plot(shap_values, X_scaled, feature_names=pipeline.FEATURE_NAMES)

In [ ]:
# Feature importance bar chart
shap.summary_plot(shap_values, X_scaled, feature_names=pipeline.FEATURE_NAMES, plot_type='bar')